In [5]:
import os
import re
import math
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor

import earthaccess
import h5py
import numpy as np
import pandas as pd
import pyproj
from tqdm import tqdm


# --- CONFIGURATION AND TARGET PARAMETERS ---

target_longitude = -68.86417
target_latitude = -20.26568
temporal_range = ("2012-01-01", "2026-11-01")
output_csv_path = "../data/processed/satellites/SDH1_VIIRS_2012_01_2026_10.csv"
temp_download_directory = "./temp_granules"

# ADDED: EXPLICIT DATASET PATHS INCLUDING QUALITY ASSURANCE BANDS FOR MA4 (1000m)
ma4_datasets_config = {
    "reflectance": [
        "HDFEOS/GRIDS/VIIRS_Grid_BRDF/Data Fields/Nadir_Reflectance_M1",
        "HDFEOS/GRIDS/VIIRS_Grid_BRDF/Data Fields/Nadir_Reflectance_M2",
        "HDFEOS/GRIDS/VIIRS_Grid_BRDF/Data Fields/Nadir_Reflectance_M3",
        "HDFEOS/GRIDS/VIIRS_Grid_BRDF/Data Fields/Nadir_Reflectance_M4",
        "HDFEOS/GRIDS/VIIRS_Grid_BRDF/Data Fields/Nadir_Reflectance_M5",
        "HDFEOS/GRIDS/VIIRS_Grid_BRDF/Data Fields/Nadir_Reflectance_M7",
        "HDFEOS/GRIDS/VIIRS_Grid_BRDF/Data Fields/Nadir_Reflectance_M8",
        "HDFEOS/GRIDS/VIIRS_Grid_BRDF/Data Fields/Nadir_Reflectance_M10",
        "HDFEOS/GRIDS/VIIRS_Grid_BRDF/Data Fields/Nadir_Reflectance_M11",
    ],
    "quality": [
        "HDFEOS/GRIDS/VIIRS_Grid_BRDF/Data Fields/BRDF_Albedo_Band_Mandatory_Quality_M1",
        "HDFEOS/GRIDS/VIIRS_Grid_BRDF/Data Fields/BRDF_Albedo_Band_Mandatory_Quality_M2",
        "HDFEOS/GRIDS/VIIRS_Grid_BRDF/Data Fields/BRDF_Albedo_Band_Mandatory_Quality_M3",
        "HDFEOS/GRIDS/VIIRS_Grid_BRDF/Data Fields/BRDF_Albedo_Band_Mandatory_Quality_M4",
        "HDFEOS/GRIDS/VIIRS_Grid_BRDF/Data Fields/BRDF_Albedo_Band_Mandatory_Quality_M5",
        "HDFEOS/GRIDS/VIIRS_Grid_BRDF/Data Fields/BRDF_Albedo_Band_Mandatory_Quality_M7",
        "HDFEOS/GRIDS/VIIRS_Grid_BRDF/Data Fields/BRDF_Albedo_Band_Mandatory_Quality_M8",
        "HDFEOS/GRIDS/VIIRS_Grid_BRDF/Data Fields/BRDF_Albedo_Band_Mandatory_Quality_M10",
        "HDFEOS/GRIDS/VIIRS_Grid_BRDF/Data Fields/BRDF_Albedo_Band_Mandatory_Quality_M11",
    ],
}

# ADDED: EXPLICIT DATASET PATHS INCLUDING QUALITY ASSURANCE BANDS FOR IA4 (500m)
ia4_datasets_config = {
    "reflectance": [
        "HDFEOS/GRIDS/VIIRS_Grid_BRDF/Data Fields/Nadir_Reflectance_I1",
        "HDFEOS/GRIDS/VIIRS_Grid_BRDF/Data Fields/Nadir_Reflectance_I2",
        "HDFEOS/GRIDS/VIIRS_Grid_BRDF/Data Fields/Nadir_Reflectance_I3",
    ],
    "quality": [
        "HDFEOS/GRIDS/VIIRS_Grid_BRDF/Data Fields/BRDF_Albedo_Band_Mandatory_Quality_I1",
        "HDFEOS/GRIDS/VIIRS_Grid_BRDF/Data Fields/BRDF_Albedo_Band_Mandatory_Quality_I2",
        "HDFEOS/GRIDS/VIIRS_Grid_BRDF/Data Fields/BRDF_Albedo_Band_Mandatory_Quality_I3",
    ],
}


# --- MODULAR FUNCTIONS ---

def compute_sinusoidal_tile_and_pixel(longitude, latitude, num_cells):
    """
    MODIFIED: MODULAR CALCULATION OF SINUSOIDAL TILE INDICES AND PIXEL COORDINATES.
    Computes horizontal/vertical tile identifiers and zero-based row/column positions.
    """
    earth_radius = 6371007.181
    sinusoidal_crs = f"+proj=sinu +lon_0=0 +x_0=0 +y_0=0 +R={earth_radius} +units=m +no_defs"

    transformer = pyproj.Transformer.from_crs("EPSG:4326", sinusoidal_crs, always_xy=True)
    target_sinu_x, target_sinu_y = transformer.transform(longitude, latitude)

    tile_size_meters = (2.0 * math.pi * earth_radius) / 36.0

    tile_horizontal = int((target_sinu_x + math.pi * earth_radius) // tile_size_meters)
    tile_vertical = int(((math.pi * earth_radius / 2.0) - target_sinu_y) // tile_size_meters)

    tile_upper_left_x = -math.pi * earth_radius + tile_horizontal * tile_size_meters
    tile_upper_left_y = (math.pi * earth_radius / 2.0) - tile_vertical * tile_size_meters

    pixel_size = tile_size_meters / num_cells
    pixel_col = int((target_sinu_x - tile_upper_left_x) // pixel_size)
    pixel_row = int((tile_upper_left_y - target_sinu_y) // pixel_size)

    return tile_horizontal, tile_vertical, pixel_row, pixel_col


def filter_granules_by_tile(granule_list, tile_horizontal, tile_vertical):
    """
    ADDED: FILTERS SEARCH RESULTS TO ENSURE GRANULES MATCH THE TARGET TILE EXACTLY.
    Prevents reading coordinates from overlapping neighboring tiles.
    """
    tile_identifier = f"h{tile_horizontal:02d}v{tile_vertical:02d}"
    filtered_granules = []

    for granule in granule_list:
        granule_ur = granule.get("umm", {}).get("GranuleUR", "")
        links = granule.data_links()
        link_url = links[0] if links else ""

        if tile_identifier in granule_ur or tile_identifier in link_url:
            filtered_granules.append(granule)

    return filtered_granules


def extract_date_from_filename(file_path):
    """
    ADDED: EXTRACTS CALENDAR DATE DIRECTLY FROM THE STANDARD VIIRS JULIAN FILENAME.
    Guarantees deterministic date mapping regardless of asynchronous download order.
    """
    file_name = os.path.basename(file_path)
    match = re.search(r"\.A(\d{4})(\d{3})\.", file_name)
    if match:
        year_str, doy_str = match.groups()
        return datetime.strptime(f"{year_str}{doy_str}", "%Y%j").strftime("%Y-%m-%d")
    return None


def process_single_granule(file_path, target_row, target_col, datasets_config):
    """
    MODIFIED: EXTRACTS SCALED REFLECTANCE AND INTEGER QUALITY FLAGS WITH EXPLICIT ERROR HANDLING.
    Deletes the temporary local file upon completion to conserve storage space.
    """
    date_string = extract_date_from_filename(file_path)
    if not date_string:
        if os.path.exists(file_path):
            os.remove(file_path)
        return None

    record = {"date": date_string}
    reflectance_scale_factor = 0.0001
    reflectance_fill_value = 32767
    quality_fill_value = 255

    try:
        with h5py.File(file_path, "r") as h5_file:
            # Read reflectance datasets and apply physical scaling
            for dataset_path in datasets_config.get("reflectance", []):
                column_name = dataset_path.split("/")[-1]
                if dataset_path in h5_file:
                    raw_val = h5_file[dataset_path][target_row, target_col]
                    record[column_name] = np.nan if raw_val == reflectance_fill_value else raw_val * reflectance_scale_factor
                else:
                    record[column_name] = np.nan

            # Read quality datasets as integers without scaling
            for dataset_path in datasets_config.get("quality", []):
                column_name = dataset_path.split("/")[-1]
                if dataset_path in h5_file:
                    raw_val = h5_file[dataset_path][target_row, target_col]
                    record[column_name] = np.nan if raw_val == quality_fill_value else int(raw_val)
                else:
                    record[column_name] = np.nan

    except Exception as error:
        # Avoid silent exceptions to enable transparent debugging
        print(f"Warning: Failed to process file {file_path}. Cause: {error}")
        return None
    finally:
        if os.path.exists(file_path):
            os.remove(file_path)

    return record


def extract_collection_parallel(granule_list, target_row, target_col, datasets_config, download_directory, batch_size=50):
    """
    MODIFIED: DOWNLOADS AND PROCESSES GRANULES CONCURRENTLY IN CONTROLLED BATCHES.
    """
    all_records = []

    for batch_start in tqdm(range(0, len(granule_list), batch_size), desc="Extracting granules"):
        batch_granules = granule_list[batch_start:batch_start + batch_size]
        downloaded_paths = earthaccess.download(batch_granules, local_path=download_directory)

        if not downloaded_paths:
            continue

        with ThreadPoolExecutor(max_workers=8) as executor:
            futures = [
                executor.submit(process_single_granule, file_path, target_row, target_col, datasets_config)
                for file_path in downloaded_paths
            ]
            for future in futures:
                result = future.result()
                if result:
                    all_records.append(result)

    return all_records


def merge_and_export_timeseries(ma4_records, ia4_records, output_path):
    """
    MODIFIED: MERGES DATASETS WITH EXPLICIT SUFFIXES TO PREVENT COLUMN COLLISIONS.
    """
    df_ma4 = pd.DataFrame(ma4_records)
    if not df_ma4.empty:
        df_ma4 = df_ma4.drop_duplicates(subset=["date"]).set_index("date")

    df_ia4 = pd.DataFrame(ia4_records)
    if not df_ia4.empty:
        df_ia4 = df_ia4.drop_duplicates(subset=["date"]).set_index("date")

    combined_df = df_ma4.join(df_ia4, how="outer", lsuffix="_ma4", rsuffix="_ia4").sort_index()
    combined_df.to_csv(output_path)
    print(f"Timeseries exported successfully to {output_path}")
    return combined_df


# --- MAIN EXECUTION PIPELINE ---

def main():
    # Authenticate session
    earthaccess.login()
    os.makedirs(temp_download_directory, exist_ok=True)

    # Compute spatial geometry
    tile_h, tile_v, ma4_row, ma4_col = compute_sinusoidal_tile_and_pixel(target_longitude, target_latitude, num_cells=1200)
    _, _, ia4_row, ia4_col = compute_sinusoidal_tile_and_pixel(target_longitude, target_latitude, num_cells=2400)

    print(f"Target sinusoidal tile: h{tile_h:02d}v{tile_v:02d}")
    print(f"VNP43MA4 target pixel (1000m): row={ma4_row}, col={ma4_col}")
    print(f"VNP43IA4 target pixel (500m):  row={ia4_row}, col={ia4_col}")

    # Search granules
    raw_ma4_granules = earthaccess.search_data(
        short_name="VNP43MA4",
        version="002",
        temporal=temporal_range,
        point=(target_longitude, target_latitude),
    )
    raw_ia4_granules = earthaccess.search_data(
        short_name="VNP43IA4",
        version="002",
        temporal=temporal_range,
        point=(target_longitude, target_latitude),
    )

    # Filter by sinusoidal tile
    ma4_granules = filter_granules_by_tile(raw_ma4_granules, tile_h, tile_v)
    ia4_granules = filter_granules_by_tile(raw_ia4_granules, tile_h, tile_v)

    print(f"Filtered VNP43MA4 granules to process: {len(ma4_granules)}")
    print(f"Filtered VNP43IA4 granules to process: {len(ia4_granules)}")

    # Execute batch extraction
    print("Extracting VNP43MA4...")
    ma4_records = extract_collection_parallel(
        ma4_granules, ma4_row, ma4_col, ma4_datasets_config, temp_download_directory, batch_size=64
    )

    print("Extracting VNP43IA4...")
    ia4_records = extract_collection_parallel(
        ia4_granules, ia4_row, ia4_col, ia4_datasets_config, temp_download_directory, batch_size=64
    )

    # Build and export consolidated dataframe
    merge_and_export_timeseries(ma4_records, ia4_records, output_csv_path)


if __name__ == "__main__":
    main()

Target sinusoidal tile: h11v11
VNP43MA4 target pixel (1000m): row=31, col=647
VNP43IA4 target pixel (500m):  row=63, col=1295


/opt/miniconda3/envs/SMAP/lib/python3.13/site-packages/earthaccess/results.py:348: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()


Filtered VNP43MA4 granules to process: 5369
Filtered VNP43IA4 granules to process: 5369
Extracting VNP43MA4...


Extracting granules:   0%|          | 0/84 [00:00<?, ?it/s]/opt/miniconda3/envs/SMAP/lib/python3.13/site-packages/earthaccess/store.py:838: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum(granule.size() for granule in granules) / 1024, 2)


QUEUEING TASKS | :   0%|          | 0/64 [00:00<?, ?it/s]

...

QUEUEING TASKS | :   0%|          | 0/57 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/57 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/57 [00:00<?, ?it/s]

Extracting granules: 100%|██████████| 84/84 [34:55<00:00, 24.94s/it]

Timeseries exported successfully to ../data/processed/satellites/SDH1_VIIRS_2012_01_2026_10.csv
